<a target="_blank" href="https://colab.research.google.com/github/instadeepai/jumanji/blob/main/examples/training.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [1]:
"""%%capture
!pip install --quiet -U "jumanji[train] @ git+https://github.com/instadeepai/jumanji.git@main"""

'%%capture\n!pip install --quiet -U "jumanji[train] @ git+https://github.com/instadeepai/jumanji.git@main'

In [2]:
# Install Jumanji from local path
import sys
import os

# Define the path to the local Jumanji package
jumanji_path = "/home/m-boelle/Documents/MarcInternship/jumanji/jumanji"

# Add the parent directory to Python path so we can import the package
sys.path.insert(0, os.path.dirname(jumanji_path))

# Verify the path exists
if os.path.exists(jumanji_path):
    print(f"Found Jumanji package at: {jumanji_path}")
else:
    print(f"Warning: Jumanji package not found at: {jumanji_path}")

# Import to verify installation
try:
    import jumanji

    print(f"Successfully imported Jumanji version: {jumanji.__version__}")
except ImportError as e:
    print(f"Failed to import Jumanji: {e}")

Found Jumanji package at: /home/m-boelle/Documents/MarcInternship/jumanji/jumanji
Successfully imported Jumanji version: 1.1.0


In [3]:
# @title Set up JAX for available hardware (run me) { display-mode: "form" }

import subprocess
import os

# Based on https://stackoverflow.com/questions/67504079/how-to-check-if-an-nvidia-gpu-is-available-on-my-system
try:
    subprocess.check_output("nvidia-smi")
    print("a GPU is connected.")
except Exception:
    # TPU or CPU
    if "COLAB_TPU_ADDR" in os.environ and os.environ["COLAB_TPU_ADDR"]:
        import jax.tools.colab_tpu

        jax.tools.colab_tpu.setup_tpu()
        print("A TPU is connected.")
    else:
        print("Only CPU accelerator is connected.")

Only CPU accelerator is connected.


In [4]:
import warnings

warnings.filterwarnings("ignore")

from jumanji.training.train import train
from hydra import compose, initialize

In [5]:
env = "job_shop"  # @param ['bin_pack', 'cleaner', 'connector', 'cvrp', 'game_2048', 'graph_coloring', 'job_shop', 'knapsack', 'maze', 'minesweeper', 'mmst', 'multi_cvrp', 'robot_warehouse', 'rubiks_cube', 'search_and_rescue', 'snake', 'sudoku', 'tetris', 'tsp']
agent = "random"  # @param ['random', 'a2c']

In [6]:
# @title Download Jumanji Configs (run me) { display-mode: "form" }

import os
import requests


def download_file(url: str, file_path: str) -> None:
    # Send an HTTP GET request to the URL
    response = requests.get(url)
    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        with open(file_path, "wb") as f:
            f.write(response.content)
    else:
        print("Failed to download the file.")


os.makedirs("configs", exist_ok=True)
config_url = "https://raw.githubusercontent.com/instadeepai/jumanji/main/jumanji/training/configs/config.yaml"
download_file(config_url, "configs/config.yaml")
env_url = f"https://raw.githubusercontent.com/instadeepai/jumanji/main/jumanji/training/configs/env/{env}.yaml"
os.makedirs("configs/env", exist_ok=True)
download_file(env_url, f"configs/env/{env}.yaml")

Failed to download the file.


In [7]:
with initialize(version_base=None, config_path="configs"):
    cfg = compose(
        config_name="config.yaml",
        overrides=[
            f"env={env}",
            f"agent={agent}",
            "logger.type=terminal",
            "logger.save_checkpoint=true",
        ],
    )

train(cfg)

INFO:root:{'devices': [CpuDevice(id=0)]}
INFO:root:Experiment: random_job_shop_improvement.
INFO:root:Starting logger.
INFO:root:Saving checkpoint...
INFO:root:Checkpoint saved at 'training_state'.
INFO:root:Closing logger...


KeyboardInterrupt: 